# Chapter 3 · Vimentin–Microtubule Bonds — Basic Inference

Sections 3.1–3.2 (pp. 27–39), Figures 6–9.

| Figure | Page | Content |
|---|---|---|
| Figure 5 | p. 29 | Cytosim snapshots (external simulation, not computed) |
| Figure 6 | p. 33 | Force regimes of a stretched polymer |
| Figure 7 | p. 34 | Sampled force trajectories, constant velocity |
| Figure 8 | p. 36 | 50 force trajectories, variable velocity |
| Figure 9 | p. 37 | Posterior of the bond parameters (1 and 40 trajectories) |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import lambert_thesis as lt
from lambert_thesis import bonds, plotting
from lambert_thesis.plotting import savefig

plotting.setup()
rng = lt.set_seed(lt.SEED)

## 3.1 Physical background (short)

In the experiment of Schaedel et al., a vimentin intermediate filament (IF) and a microtubule (MT) are held by beads in
optical traps, brought into perpendicular contact, and the vimentin is dragged across the microtubule at constant
velocity $v$. Occasionally a bond forms at the crossing point; the force on it is measured until it breaks.
The bond unbinds with the Bell–Evans rate

$$r_u(t)=r_{u,0}\cdot\exp\left(\frac{F(t)\,x_u}{k_BT}\right),$$

with base unbinding rate $r_{u,0}$ and sensitivity $x_u$ — the **bond parameters** we want to infer.
Slow driving probes $r_{u,0}$; fast driving produces large forces and probes $x_u$.

## 3.2.1 Data generation

The force on the bond is caused by the stress in the stretched IF. In the *entropic regime* the polymer is
entangled by thermal fluctuations; when strongly stretched, the internal bonds themselves are strained
(*Hookean regime*). The force is implicitly given by

$$\frac{x}{d_\text{IF}}=\coth\left(\frac{2L_PF}{k_BT}\right)-\frac{k_BT}{2L_PF},$$

with persistence length $L_P$ and contour length $d_\text{IF}$ ($1/d_\text{IF}$ acts as the hardness of the spring).
For the simulation we sample solutions of this equation and fit a Chebyshev polynomial, which gives a fast explicit
approximation.

*Code:* `bonds.force_curve(k)` follows the entropic relation until its slope reaches the spring constant `k`
and continues linearly beyond (Hookean regime); the samples are fitted with `numpy.polynomial.Chebyshev`.
Parameters: $L_P=1.5$, $k_BT=1$ (arbitrary units).

In [ ]:
x = np.linspace(0, 1, 200)
fig, ax = plt.subplots(figsize=(5, 4))
for k in np.linspace(1, 200, 30):
    ax.plot(x, bonds.force_curve(k)(x), "r", lw=1.5)
ax.plot([], [], "r", label="force curves for 30 spring constants\n$k\\in[1, 200]$ (harder = steeper)")
ax.text(0.02, 0.8, "Entropic regime", fontsize=12, rotation=4)
ax.text(0.88, 7, "Hookean Regime", fontsize=12, rotation=90)
ax.set(xlabel="Stretching distance", ylabel="Force")
ax.legend(loc="upper left")
savefig("fig06_force_regimes");

**Figure 6 (p. 33).** Force inside a stretched polymer vs. the direct distance of the endpoints (arbitrary units).
Decreasing the contour length increases the hardness, i.e. how quickly the force grows in reaction to the strain.

**Simulation.** At each time $t$ we imprint the strain $v\,t$, compute $F(t)$ and the unbinding rate $r_u(t)$.
Assuming quasi-constant stress between two measurements $\Delta t$ apart, the bond breaks with probability

$$p_u=1-\exp\left(-\Delta t\cdot r_{u,0}\exp\left(\frac{F(t)\,x_u}{k_BT}\right)\right).$$

If it breaks, the trajectory ends; otherwise time advances by $\Delta t$. Since $\Delta t\,r_u\ll1$, the probability is
evaluated to first order, $p_u=\min(1,\Delta t\,r_u)$, in the simulation as well as in the likelihood below.

*Code:* `bonds.simulate_trajectory(force, ru0, xu, v, dt, rng)`. Ground truth: $r_{u,0}=0.05$, $x_u=0.2$;
spring constant $k=50$; $\Delta t=10^{-2}$; for Figure 7 a constant velocity $v=0.2$.

In [ ]:
RU0_GT, XU_GT = 0.05, 0.2
DT = 1e-2
force = bonds.force_curve(50)

trajs_const, _ = bonds.simulate_trajectories(force, RU0_GT, XU_GT, v=0.2, dt=DT, rng=rng, n=50)

fig, ax = plt.subplots(figsize=(5.5, 4.5))
for i, F in enumerate(trajs_const):
    ax.plot(F + i)
ax.set(title="Simulated Trajectories (offset for visibility)", xlabel="Timesteps", ylabel="Force")
ax.text(0.98, 0.02, "each colour: one trajectory, shifted up by its index;\nthe line ends where the bond broke",
        transform=ax.transAxes, ha="left", va="top", fontsize=8)
savefig("fig07_simple_forcecurves");

**Figure 7 (p. 34).** 50 sampled trajectories of the simple entropic spring model at constant velocity.
Time is proportional to the strain. The curves are shifted upwards to make them distinguishable.
Schaedel et al. only evaluated the bond *lifetimes* with a KS test; the Bayesian inference uses the measured forces directly.

## 3.2.2 Likelihood

Each data point carries the force $F_i$ and the bond state $S_i$: bound ($s_i$) or unbound ($\bar s_i$).
Since the bond has no memory, all observations are independent and

$$L(F_i, s_i\mid r_{u,0},x_u)=\exp\left(-\Delta t\, r_{u,0}e^{F_ix_u/k_BT}\right),\qquad
L(F_i, \bar s_i\mid r_{u,0},x_u)=1-\exp\left(-\Delta t\, r_{u,0}e^{F_ix_u/k_BT}\right),$$

$$\mathcal L(F_{1:i},S_{1:i}\mid r_{u,0},x_u)=\prod_i L(F_i,S_i\mid r_{u,0},x_u,S_{1:i-1}).$$

## 3.2.3 Posterior density

With a flat prior, $p(r_{u,0},x_u\mid F,S)\propto\mathcal L$. We evaluate it on a grid that is broad enough to capture
the posterior mass (the grid also sets the prior range) and normalise.
The data for the inference use a different velocity for every trajectory, $v\sim\mathcal U(0.01, 0.4)$.

*Code:* `bonds.log_likelihood_grid(trajectories, ru0_grid, xu_grid, dt)`. In log space the product becomes a sum
over all data points, evaluated for every grid point at once with array operations.

In [ ]:
trajs_var, v_var = bonds.simulate_trajectories(force, RU0_GT, XU_GT, v=lambda r: r.uniform(0.01, 0.4),
                                               dt=DT, rng=rng, n=50)

fig, ax = plt.subplots(figsize=(4, 4))
for F in trajs_var:
    ax.plot(F)
ax.set(xlabel="Timesteps", ylabel="Force")
ax.text(0.97, 0.97, "50 trajectories (one colour each),\n$v\\sim\\mathcal{U}(0.01, 0.4)$", transform=ax.transAxes,
        ha="right", va="top", fontsize=8)
savefig("fig08_forces_variable_v");

**Figure 8 (p. 36).** The 50 force trajectories used for the inference with variable driving velocity. Shallow
trajectories produce more data in the entropic regime, steep ones inform the sensitivity.

In [ ]:
ru0_grid = np.linspace(0, 2.2 * RU0_GT, 100)
xu_grid = np.linspace(0, 2.2 * XU_GT, 100)
extent = (ru0_grid[0], ru0_grid[-1], xu_grid[0], xu_grid[-1])

def plot_posterior(trajs, ax=None, mass=0.95):
    """Grid posterior with its 95 % highest-density region and the ground truth."""
    P = bonds.normalize_log(bonds.log_likelihood_grid(trajs, ru0_grid, xu_grid, DT))
    ax = ax or plt.gca()
    ax.imshow(P, origin="lower", extent=extent, aspect="auto", cmap="ocean_r")
    ax.contour(ru0_grid, xu_grid, P, levels=[bonds.hdi_threshold(P, mass)], linestyles="--", colors="k", alpha=0.8)
    ax.plot([RU0_GT], [XU_GT], "rx", markersize=10, markeredgewidth=2)
    ax.plot([], [], "k--", label=f"{mass:.0%} highest-density region")
    ax.plot([], [], "rx", label="ground truth")
    ax.set(title=f"Posteriors, {len(trajs)} trajectories, Dashed: {mass*100:.0f}th percentile",
           xlabel="ru0", ylabel="xu", xlim=extent[:2], ylim=extent[2:])
    ax.legend(loc="upper right", fontsize=8)
    return P

### Figure 9 (p. 37), left: a single trajectory

In [ ]:
plt.figure()
plot_posterior(trajs_var[:1])
savefig("fig09a_posterior_1_trajectory");

### Figure 9 (p. 37), right: 40 trajectories
The posterior uses the first 40 of the 50 trajectories of Figure 8.

In [ ]:
plt.figure()
P40 = plot_posterior(trajs_var[:40])
savefig("fig09b_posterior_40_trajectories");

**Figure 9 (p. 37).** Posterior density of the bond parameters for synthetic observations; colour = posterior density,
red cross = ground truth. A single trajectory admits a wide range of bond parameters; more data narrows the posterior
towards the ground truth. Increasing the number of observations is the single most influential factor for the precision.